In [ ]:
# Assistants API 
# - Maintains conversation context dynamically
# - Stateful, keeps track of interactions
# - Supports function calling

# Code Interpreter 
# - Enables the assistance API to handle and modify Python code iteratively
# - Useful for complex tasks , adjusting code until it succeeds
# - Functions as an environment within your current environment for autonomous code modification. This allows the 
# agent to modify its own code

In [52]:
%run interact_csv_and_sql_data/sql_helpers.py

In [53]:
import json
import os 

from openai import AzureOpenAI

client = AzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# I create assistant - Create the AI worker and tell it what job it should do
assistant = client.beta.assistants.create(
    instructions="You are an assistant answering questions about a Covid dataset.",
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT") ,
    tools=tools_sql
)


# II Create thread - will create traces of discussion so we can connect different messages
# A thread is the conversation container. It stores the history of interaction between: user, assistant, tool outputs. 
# In plain English, Create a chat session where the conversation will live. You can think of it like opening a new conversation window

    
thread = client.beta.threads.create()
print(thread)

# unique identifier for the thread created

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/765077753.py:12: DeprecationWarning: deprecated
  assistant = client.beta.assistants.create(


Thread(id='thread_LwhPwmL2HSQGF10Kz34IXLaj', created_at=1780925980, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/765077753.py:30: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create()


In [54]:
# III - add message (from user's perspective)
# This means adding the user’s question into the thread. At this point, the assistant still has not answered yet. You have only added the message.

messages = client.beta.threads.messages.create(
  thread_id=thread.id,
    role="user",
    content="how many hospitalized people we had in Alaska on 2021-03-05"
)

print(messages.model_dump_json(indent=2))

{
  "id": "msg_916tE49dAVFxBRaoE3UAbigO",
  "assistant_id": null,
  "attachments": [],
  "completed_at": null,
  "content": [
    {
      "text": {
        "annotations": [],
        "value": "how many hospitalized people we had in Alaska on 2021-03-05"
      },
      "type": "text"
    }
  ],
  "created_at": 1780925983,
  "incomplete_at": null,
  "incomplete_details": null,
  "metadata": {},
  "object": "thread.message",
  "role": "user",
  "run_id": null,
  "status": null,
  "thread_id": "thread_LwhPwmL2HSQGF10Kz34IXLaj"
}


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/485511856.py:4: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.create(


In [55]:
# see the same message in a different format

messages = client.beta.threads.messages.list(
  thread_id=thread.id
)

print(messages.model_dump_json(indent=2))

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/278723701.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


{
  "data": [
    {
      "id": "msg_916tE49dAVFxBRaoE3UAbigO",
      "assistant_id": null,
      "attachments": [],
      "completed_at": null,
      "content": [
        {
          "text": {
            "annotations": [],
            "value": "how many hospitalized people we had in Alaska on 2021-03-05"
          },
          "type": "text"
        }
      ],
      "created_at": 1780925983,
      "incomplete_at": null,
      "incomplete_details": null,
      "metadata": {},
      "object": "thread.message",
      "role": "user",
      "run_id": null,
      "status": null,
      "thread_id": "thread_LwhPwmL2HSQGF10Kz34IXLaj"
    }
  ],
  "has_more": false,
  "object": "list",
  "first_id": "msg_916tE49dAVFxBRaoE3UAbigO",
  "last_id": "msg_916tE49dAVFxBRaoE3UAbigO"
}


In [56]:
# IV) Run assistant on 
# This tells the assistant: read the conversation in the thread, decide what to do, possibly call tools/functions, generate an answer. 
# This is the step where the assistant actually starts working

run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id,
)

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2009769195.py:5: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


In [57]:
from pathlib import Path
from sqlalchemy import create_engine
import pandas as pd

db_path = Path("interact_csv_and_sql_data")/ "db" / "test.db"

db_path.parent.mkdir(parents=True, exist_ok=True)
engine = create_engine(f'sqlite:///{db_path.resolve()}')

In [58]:
# Leverage the function calling with Assistants API

import time
from IPython.display import clear_output

start_time = time.time()

status = run.status

while status not in ["completed", "cancelled", "expired", "failed"]:
    time.sleep(5)

    # running a thread on an ongoing discussion
    run = client.beta.threads.runs.retrieve(
        thread_id=thread.id,run_id=run.id
    )
    print("Elapsed time: {} minutes {} seconds".format(
        int((time.time() - start_time) // 60),
        int((time.time() - start_time) % 60))
         )
    status = run.status
    print(f'Status: {status}')

    # requires_action appears when the assistant has decided to call one of your custom functions and is waiting for your notebook to execute 
    # it and return the output.
    if (status=="requires_action"):
        available_functions = {
            "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
            "get_hospitalized_increase_for_state_on_date":get_hospitalized_increase_for_state_on_date
        }

        tool_outputs = []
        for tool_call in run.required_action.submit_tool_outputs.tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(
                engine=engine,
                state_abbr=function_args.get("state_abbr"),
                specific_date=function_args.get("specific_date"),
            )
            print(function_response)
            print(tool_call.id)
            
            tool_outputs.append(
                { "tool_call_id": tool_call.id,
                 "output": str(function_response)
                }
            )

        run = client.beta.threads.runs.submit_tool_outputs(
          thread_id=thread.id,
          run_id=run.id,
          tool_outputs = tool_outputs
        )


messages = client.beta.threads.messages.list(
  thread_id=thread.id
)

print(messages)

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2585822993.py:14: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


Elapsed time: 0 minutes 5 seconds
Status: requires_action
{'date': '2021-03-05', 'hospitalizedIncrease': 3}
call_P5hz2PU1hyjgufnXEYPdy2CM


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2585822993.py:48: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.submit_tool_outputs(


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2585822993.py:14: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


Elapsed time: 0 minutes 11 seconds
Status: completed


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2585822993.py:55: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


SyncCursorPage[Message](data=[Message(id='msg_y1Ma818ZxI4Du3LM5QJqrJYH', assistant_id='asst_jfPzzqExE1IGUun41pAfkW4G', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='In Alaska on 2021-03-05, the hospitalized increase was 3.'), type='text')], created_at=1780926009, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='assistant', run_id='run_1fqiS0EQUgnMvUNQkhQdOOmR', status=None, thread_id='thread_LwhPwmL2HSQGF10Kz34IXLaj'), Message(id='msg_916tE49dAVFxBRaoE3UAbigO', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='how many hospitalized people we had in Alaska on 2021-03-05'), type='text')], created_at=1780925983, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_LwhPwmL2HSQGF10Kz34IXLaj')], has_more=False, object='list', first_id='msg_y1Ma818ZxI4Du3LM5Q

In [67]:
# Add the code interpreter

# Code Interpreter gives the assistant a temporary Python-like working environment where it can:

# open attached files
# inspect data
# write and run code
# use that code to answer your question
# So instead of calling your custom Python function, the assistant can analyze the uploaded CSV on 
# its own.



# Upload my CSV file so the assistant can use it.
file = client.files.create(
  file=open("interact_csv_and_sql_data/all-states-history.csv", "rb"),
  purpose='assistants'
)

# Create assistant
# Create an assistant that can work with the uploaded file using Code Interpreter resources
assistant = client.beta.assistants.create(
  instructions="You are an assitant answering questions about a Covid dataset.",  
  model=os.getenv("AZURE_OPENAI_DEPLOYMENT"), 
  tools=[{"type": "code_interpreter"}],
  #tool_resources={"code_interpreter": {"file_ids" : [file.id]}}
)

# Create thread - start a new conversation
thread = client.beta.threads.create()
print(thread)


# Add message - Ask the assistant the question
# attach the uploaded file to the message 
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    #content="how many hospitalized people we had in Alaska the 2021-03-05?",
    content="Using the uploaded dataset, tell me the value of hospitalizedIncrease for Alaska (AK) on 2021-03-05.",
    attachments= [
        {
            "file_id": file.id,
            "tools": [{"type": "code_interpreter"}]
        }
    ]
               
)
print(message)

# Run assistant on thread - the assistant will process the question
run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id,
)

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2083030609.py:13: DeprecationWarning: deprecated
  assistant = client.beta.assistants.create(


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2083030609.py:28: DeprecationWarning: deprecated
  assistant = client.beta.assistants.create(
/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2083030609.py:36: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create()


Thread(id='thread_J8F2kUmsnHKQC4CfN2cyhrh0', created_at=1780929528, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2083030609.py:42: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(


Message(id='msg_HR1XvgIDAcTO1yJRCdQtx7SQ', assistant_id=None, attachments=[Attachment(file_id='assistant-KMQwEJgXfVdfEwnoiTH6um', tools=[CodeInterpreterTool(type='code_interpreter')])], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='Using the uploaded dataset, tell me the value of hospitalizedIncrease for Alaska (AK) on 2021-03-05.'), type='text')], created_at=1780929529, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_J8F2kUmsnHKQC4CfN2cyhrh0')


/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/2083030609.py:58: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


In [68]:
status = run.status
start_time = time.time()
while status not in ["completed", "cancelled", "expired", "failed"]:
    time.sleep(5)
    run = client.beta.threads.runs.retrieve(
        thread_id=thread.id,
        run_id=run.id
    )
    print("Elapsed time: {} minutes {} seconds".format(
        int((time.time() - start_time) // 60),
        int((time.time() - start_time) % 60))
         )
    status = run.status
    print(f'Status: {status}')
    clear_output(wait=True)


messages = client.beta.threads.messages.list(
  thread_id=thread.id
)

print(messages.model_dump_json(indent=2))

/var/folders/yl/tx2xl26s30qbvj6_cb5bgw1w0000gp/T/ipykernel_27593/3504059192.py:18: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


{
  "data": [
    {
      "id": "msg_8Q0rFBd6kly6NPJVmxVF2NWY",
      "assistant_id": "asst_v2ThiTYGkM18AwXxCVUkzMfM",
      "attachments": [],
      "completed_at": null,
      "content": [
        {
          "text": {
            "annotations": [],
            "value": "The value of **hospitalizedIncrease** for **Alaska (AK)** on **2021-03-05** is **3**."
          },
          "type": "text"
        }
      ],
      "created_at": 1780929539,
      "incomplete_at": null,
      "incomplete_details": null,
      "metadata": {},
      "object": "thread.message",
      "role": "assistant",
      "run_id": "run_Zu26FzpGs4pl6y5CFPsm32ZP",
      "status": null,
      "thread_id": "thread_J8F2kUmsnHKQC4CfN2cyhrh0"
    },
    {
      "id": "msg_HR1XvgIDAcTO1yJRCdQtx7SQ",
      "assistant_id": null,
      "attachments": [
        {
          "file_id": "assistant-KMQwEJgXfVdfEwnoiTH6um",
          "tools": [
            {
              "type": "code_interpreter"
            }
          ]
    